In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

np.random.seed(109204)

# ---- For Logistic ----

def logistic_map(x, r): # Function for Logistic
    return r * x * (1 - x)
plt.rcParams.update({'axes.labelsize': 18, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.titlesize': 18,})

sigma_Q = 0.5  # Controls Variability of Q_k
def ekf_logistic_r_estimation(r_true, r_guess, n_steps=100, Q=1e-5, Q_seq=0.5, x0_true=0.7, x0_hat=0.7, y_meas=None):
    generated = False
    Q0 = Q
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps))    
    if y_meas is None:
        generated = True
        x_true = np.zeros(n_steps)
        x_true[0] = x0_true
        y_meas[0] = x_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        for k in range(1, n_steps):
            x_true[k] = logistic_map(x_true[k-1], r_true)  
            y_meas[k] = x_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    else:
        x_true = None # If data does already exist, just use that

    x_hat = np.zeros(n_steps) # EKF Setup
    r_hat = np.zeros(n_steps)
    x_hat[0] = x0_hat
    r_hat[0] = r_guess
    P = np.eye(2)
    C = np.array([[1, 0]])

    for k in range(1, n_steps):
        x_prev, r_prev = x_hat[k-1], r_hat[k-1]

        x_pred = logistic_map(x_prev, r_prev) # Prediction Stage
        r_pred = r_prev
        X_pred = np.array([[x_pred], [r_pred]])

        dphidx = r_prev * (1 - 2*x_prev) # Local Linearisation
        dphidr = x_prev * (1 - x_prev)
        Phi = np.array([[dphidx, dphidr], [0, 1]])
        P_pred = Phi @ P @ Phi.T

        y_pred = (C @ X_pred)[0, 0] # Update Measurement
        innovation = y_meas[k] - y_pred
        S = (C @ P_pred @ C.T)[0, 0] + Q_seq[k]

        K = (P_pred @ C.T) / S 
        X_upd = X_pred + K * innovation
        P = (np.eye(2) - K @ C) @ P_pred # Non-Optimal Update (not Joseph Form)

        X_upd = np.asarray(X_upd).reshape(-1)
        x_hat[k], r_hat[k] = X_upd[0], X_upd[1]
    return x_true, y_meas, x_hat, r_hat

def interactive_ekf(r_true=2.8, Q=1e-5, n_steps = 100):
    r_guess = 1
    n_runs = 100
    Q0 = Q
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps)) # True Logistic Map (Single Trajectory)
    x_true_base = np.zeros(n_steps)
    y_meas_base = np.zeros(n_steps)
    x_true_base[0] = 0.7
    y_meas_base[0] = x_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    for k in range(1, n_steps):
        x_true_base[k] = logistic_map(x_true_base[k-1], r_true)
        y_meas_base[k] = x_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))

    all_x_true = np.zeros((n_runs, n_steps))
    all_x_hat = np.zeros((n_runs, n_steps))
    all_r_hat = np.zeros((n_runs, n_steps))
    for i in range(n_runs):
        Q_seq = Q * np.exp(sigma_Q * np.random.randn(n_steps))
        y_meas = x_true_base + np.random.normal(0, np.sqrt(Q_seq), size=n_steps)
        x_true, y_meas, x_hat, r_hat = ekf_logistic_r_estimation(r_true = r_true, r_guess=r_guess, n_steps=n_steps, Q=Q, x0_true=0.7, x0_hat=0.7, y_meas=y_meas, Q_seq=Q_seq)
        all_x_true[i, :] = x_true_base
        all_x_hat[i, :] = x_hat
        all_r_hat[i, :] = r_hat

    x_hat_mean = np.mean(all_x_hat, axis=0) # Statistics from MC
    x_hat_p05 = np.percentile(all_x_hat, 5, axis=0)
    x_hat_p95 = np.percentile(all_x_hat, 95, axis=0)  
    mse_t = np.mean((all_x_hat - all_x_true)**2, axis=0) # MSE of MC
    variance_t = np.var(all_x_hat, axis=0) # Filter Spread (MC Variance)

    fig, ax = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
    ax[0, 0].plot(x_true_base, 'k-', lw=2, label='True Logistic Map')
    ax[0, 0].plot(x_hat_mean, 'b--', lw=1.5, label='Mean EKF Estimate')
    ax[0, 0].plot(y_meas_base, 'r.', alpha=0.4, label='Measurements')
    ax[0, 0].fill_between(range(n_steps), x_hat_p05, x_hat_p95, alpha=0.2, label='95% Confidence Band')
    ax[0, 0].set_title(f'EKF Monte Carlo (r_true={r_true:.3f}, Q={Q:.1e})')
    ax[0, 0].set_ylabel('x')
    ax[0, 0].legend()

    r_hat_mean = np.mean(all_r_hat, axis=0)
    r_hat_p05 = np.percentile(all_r_hat, 5, axis=0)
    r_hat_p95 = np.percentile(all_r_hat, 95, axis=0)
    ax[0, 1].axhline(r_true, color='k', lw=2, label='True r')
    ax[0, 1].plot(r_hat_mean, 'b--', lw=1.5, label='Mean EKF Estimate')
    ax[0, 1].fill_between(range(n_steps), r_hat_p05, r_hat_p95, alpha=0.2, label='95% Confidence Band')
    ax[0, 1].set_title('EKF Parameter Evolution Across MC Runs')
    ax[0, 1].set_ylabel('r')
    ax[0, 1].legend()

    ax[1, 0].plot(mse_t, 'b-', label='MSE')
    ax[1, 0].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 0].set_xlabel('Iteration')
    ax[1, 0].set_ylabel('MSE')
    ax[1, 0].set_title('MSE Evolution Over Time')
    ax[1, 0].legend()

    ax[1, 1].plot(variance_t, 'r-', label='MC Variance')
    ax[1, 1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 1].set_xlabel('Iteration')
    ax[1, 1].set_ylabel('Variance')
    ax[1, 1].set_title('Filter Spread of MC')
    ax[1, 1].legend()

    plt.tight_layout()
    plt.show()

interact( # Interactive Sliders
    interactive_ekf,
    r_true=FloatSlider(value=2.8, min=1, max=4, step=0.01, description='True r'),
    Q=FloatSlider(value=1e-5, min=1e-15, max=1e-2, step=1e-6, readout_format='.1e', description='Q'),
    n_steps=IntSlider(value=100, min=100, max=500, step=50, description='Iterations'))

interactive(children=(FloatSlider(value=2.8, description='True r', max=4.0, min=1.0, step=0.01), FloatSlider(v…

<function __main__.interactive_ekf(r_true=2.8, Q=1e-05, n_steps=100)>